In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/TabNet'
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"Folder Created: {PROJECT_PATH}")
else:
    print(f"Folder already exists: {PROJECT_PATH}")

os.makedirs(f'{PROJECT_PATH}/data', exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/models', exist_ok=True)
%cd {PROJECT_PATH}

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder already exists: /content/drive/MyDrive/TabNet
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3179/3898122059.py", line 15, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{PROJECT_PATH}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    o

1. 주요 Requirements 설치

In [2]:
!pip install torch pytorch-tabnet scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.5 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, classification_report,
    precision_score, recall_score, f1_score, average_precision_score, log_loss,
    roc_curve, precision_recall_curve, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# ==========================================
# 1. 데이터 로더 정의
# ==========================================
def get_hr_data_for_tabnet(filepath):
    df = pd.read_csv(filepath)

    target_col = 'Attrition'
    X = df.drop(target_col, axis=1)
    y = LabelEncoder().fit_transform(df[target_col]) # 'Yes'/'No' -> 1/0

    #데이터 분할 (Train/Valid/Test = 70:15:15)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # 변수 타입 분류
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

    # 수치형 스케일링
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[num_cols])
    X_valid_num = scaler.transform(X_valid[num_cols])
    X_test_num = scaler.transform(X_test[num_cols])

    # 범주형 오디널 인코딩
    ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_cat = ordinal_enc.fit_transform(X_train[cat_cols])
    X_valid_cat = ordinal_enc.transform(X_valid[cat_cols])
    X_test_cat = ordinal_enc.transform(X_test[cat_cols])

    # numpy 배열 합치기 (수치형 컬럼 먼저, 범주형 컬럼 나중)
    X_train_processed = np.hstack((X_train_num, X_train_cat))
    X_valid_processed = np.hstack((X_valid_num, X_valid_cat))
    X_test_processed = np.hstack((X_test_num, X_test_cat))

    # TabNet용 메타데이터 계산
    features = num_cols + cat_cols
    cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
    cat_dims = [len(cats) for cats in ordinal_enc.categories_]

    return X_train_processed, X_valid_processed, X_test_processed, y_train, y_valid, y_test, features, cat_idxs, cat_dims

# ==========================================
# 2. 데이터 불러오기
# ==========================================
filepath = '/content/drive/MyDrive/data_team7.csv'
X_train, X_valid, X_test, y_train, y_valid, y_test, features, cat_idxs, cat_dims = get_hr_data_for_tabnet(filepath)

# ==========================================
# 3. TabNet 모델 정의 및 학습
# ==========================================
clf = TabNetClassifier(
    n_d=16, n_a=16, n_steps=3,
    gamma=1.2866,
    lambda_sparse=0.0004,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.0275),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size":10, "gamma":0.9},
    mask_type='sparsemax',
    device_name='auto'
)

print("🚀 TabNet 학습을 시작합니다...")
clf.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=100,
    batch_size=256,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# ==========================================
# 4. 모델 평가 및 피처 중요도 확인
# ==========================================
preds = clf.predict(X_test)
preds_proba = clf.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, preds_proba)
pr_auc = average_precision_score(y_test, preds_proba)
logloss = log_loss(y_test, preds_proba)

print("\n🏆 [최종 모델 성능 평가 지표 (6 Metrics)]")
print("-" * 40)
print(f"1. Precision: {precision:.4f} | 2. Recall: {recall:.4f} | 3. F1-Score: {f1:.4f}")
print(f"4. ROC-AUC  : {roc_auc:.4f} | 5. PR-AUC: {pr_auc:.4f} | 6. Log Loss: {logloss:.4f}")
print("-" * 40)

feat_importances = clf.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feat_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# ==========================================
# 5. 시각화 대시보드 및 PDF 저장
# ==========================================
pdf_filepath = '/content/drive/MyDrive/result/TabNet_Evaluation_Report_f1_optimalparam_epoch200.pdf'
sns.set_style("whitegrid")

with PdfPages(pdf_filepath) as pdf:
    # --- 대시보드 ---
    fig1 = plt.figure(figsize=(18, 12))
    fig1.suptitle("TabNet Performance Dashboard", fontsize=20, fontweight='bold')

    ax1 = plt.subplot(2, 2, 1)
    ax1.plot(clf.history['train_auc'], label='Train AUC', color='blue', lw=2)
    ax1.plot(clf.history['valid_auc'], label='Validation AUC', color='orange', lw=2)
    ax1.set_title("Learning Curve (AUC)", fontsize=14, fontweight='bold')
    ax1.legend()

    ax2 = plt.subplot(2, 2, 2)
    fpr, tpr, _ = roc_curve(y_test, preds_proba)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_title("ROC Curve", fontsize=14, fontweight='bold')
    ax2.legend()

    ax3 = plt.subplot(2, 2, 3)
    precisions, recalls, _ = precision_recall_curve(y_test, preds_proba)
    ax3.plot(recalls, precisions, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    baseline = sum(y_test) / len(y_test)
    ax3.plot([0, 1], [baseline, baseline], linestyle='--', color='gray')
    ax3.set_title("Precision-Recall Curve", fontsize=14, fontweight='bold')
    ax3.legend()

    ax4 = plt.subplot(2, 2, 4)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4, annot_kws={"size": 14})
    ax4.set_title("Confusion Matrix", fontsize=14, fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pdf.savefig(fig1)
    plt.show()
    plt.close(fig1)

    # --- 피처 중요도 ---
    fig2 = plt.figure(figsize=(12, 10))
    sns.barplot(x='Importance', y='Feature', data=importance_df.head(20), palette='viridis')
    plt.title("Top 20 Feature Importances in TabNet", fontsize=16, fontweight='bold')
    plt.tight_layout()

    pdf.savefig(fig2)
    plt.show()
    plt.close(fig2)

print(f"\n✅ [완료] 그래프가 성공적으로 PDF로 저장되었습니다: {pdf_filepath}")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3179/4012275787.py", line 1, in <cell line: 0>
    import pandas as pd
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceb

In [5]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 30.2 MB/s eta 0:00:00


In [7]:
import optuna
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

# 사용 가능한 옵션: 'roc_auc', 'pr_auc', 'f1', 'recall'
TARGET_METRIC = 'pr_auc'
# ==========================================
# 1. 목적 함수(Objective Function) 정의
# ==========================================
def objective(trial):
    # 1. 모델 복잡도 (데이터가 적으므로 보수적으로 설정)
    n_steps = trial.suggest_int('n_steps', 2, 5)
    n_d = trial.suggest_int('n_d', 8, 32, step=8) # 8, 16, 24, 32중 선택
    n_a = n_d

    # 2. 피처 재사용률과 희소성 규제
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lambda_sparse = trial.suggest_float('lambda_sparse', 1e-4, 1e-2, log=True)

    # 3. 학습률 (Learning Rate) 튜닝 추가
    lr = trial.suggest_float('lr', 0.005, 0.05, log=True)

    # 🤖 모델 정의 (Sparsemax는 전략적으로 고정)
    clf = TabNetClassifier(
        n_d=n_d, n_a=n_a, n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=2,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size":10, "gamma":0.9},
        mask_type='sparsemax',
        device_name='auto',
        verbose=0
    )

    # 🚀 모델 학습 (Train 데이터로 학습, Valid 데이터로 평가)
    clf.fit(
        X_train=X_train, y_train=y_train,
        eval_set=[(X_valid, y_valid)],
        eval_name=['valid'],
        eval_metric=['auc'],
        max_epochs=100,
        patience=20,
        batch_size=256,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )
    if TARGET_METRIC in ['roc_auc', 'pr_auc']:
        # 확률값(0.0 ~ 1.0)이 필요한 지표들
        preds_proba = clf.predict_proba(X_valid)[:, 1]

        if TARGET_METRIC == 'roc_auc':
            score = roc_auc_score(y_valid, preds_proba)
        elif TARGET_METRIC == 'pr_auc':
            score = average_precision_score(y_valid, preds_proba)

    elif TARGET_METRIC in ['f1', 'recall']:
        # 확정된 클래스(0 또는 1)가 필요한 지표들
        preds = clf.predict(X_valid)

        if TARGET_METRIC == 'f1':
            score = f1_score(y_valid, preds)
        elif TARGET_METRIC == 'recall':
            score = recall_score(y_valid, preds)

    return score

print("⏳ Optuna 하이퍼파라미터 튜닝을 시작합니다...")

# ==========================================
# 3. 결과 확인 및 베스트 모델 훈련
# ==========================================
study = optuna.create_study(direction='maximize', study_name=f"TabNet_{TARGET_METRIC}_Tuning")
study.optimize(objective, n_trials=100)

print(f"\n🎉 [튜닝 완료] {TARGET_METRIC} 기준 최고 점수: {study.best_value:.4f}")
print("베스트 파라미터:", study.best_params)

[I 2026-05-27 04:12:15,953] A new study created in memory with name: TabNet_pr_auc_Tuning


⏳ Optuna 하이퍼파라미터 튜닝을 시작합니다...


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-05-27 04:12:21,746] Trial 0 finished with value: 0.20293690366074377 and parameters: {'n_steps': 2, 'n_d': 8, 'gamma': 1.790970780381392, 'lambda_sparse': 0.001947308156692035, 'lr': 0.0058702110870917125}. Best is trial 0 with value: 0.20293690366074377.



Early stopping occurred at epoch 48 with best_epoch = 28 and best_valid_auc = 0.58409


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-05-27 04:12:28,434] Trial 1 finished with value: 0.4266994946575049 and parameters: {'n_steps': 2, 'n_d': 24, 'gamma': 1.2556819182488956, 'lambda_sparse': 0.0002915192420212415, 'lr': 0.034991977103675985}. Best is trial 1 with value: 0.4266994946575049.


...


Early stopping occurred at epoch 26 with best_epoch = 6 and best_valid_auc = 0.70394


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-05-27 04:26:03,187] Trial 98 finished with value: 0.31704469772456284 and parameters: {'n_steps': 3, 'n_d': 24, 'gamma': 1.52021800029441, 'lambda_sparse': 0.0002503254173144643, 'lr': 0.023013741232858122}. Best is trial 58 with value: 0.5961719700691316.



Early stopping occurred at epoch 44 with best_epoch = 24 and best_valid_auc = 0.7078


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-05-27 04:26:11,745] Trial 99 finished with value: 0.36205501691942826 and parameters: {'n_steps': 3, 'n_d': 8, 'gamma': 1.4571283140991305, 'lambda_sparse': 0.00029990162301768187, 'lr': 0.029831029773353702}. Best is trial 58 with value: 0.5961719700691316.



Early stopping occurred at epoch 56 with best_epoch = 36 and best_valid_auc = 0.75058

🎉 [튜닝 완료] pr_auc 기준 최고 점수: 0.5962
베스트 파라미터: {'n_steps': 3, 'n_d': 16, 'gamma': 1.286578826322205, 'lambda_sparse': 0.0003792860210675046, 'lr': 0.027536609006609095}
